# 🏆 Week 4 · Day 22-23: 실전 프로젝트 — Titanic End-to-End

## 🎯 학습 목표
- 지금까지 배운 모든 것을 **하나의 대회**에 적용
- EDA → FE → CV → Model → Ensemble → Submit 전체 파이프라인
- **실제 제출 파일** 생성
- Kaggle Getting Started 대회 Top 5% 전략

> 📌 **대회 링크**: https://www.kaggle.com/c/titanic

## 데이터 다운로드

```bash
kaggle competitions download -c titanic -p ../data/
unzip ../data/titanic.zip -d ../data/titanic/
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import warnings; warnings.filterwarnings('ignore')
pd.options.display.max_columns = 30
np.random.seed(42)

## 1. 데이터 로드

실제 Titanic 데이터 대신 여기서는 **재현 가능한 샘플**을 생성합니다. 실제 대회에서는:
```python
train = pd.read_csv('../data/titanic/train.csv')
test  = pd.read_csv('../data/titanic/test.csv')
```


In [ ]:
# Titanic 데이터 재현 (실제 대회 데이터와 동일 스키마)
def make_titanic():
    np.random.seed(42)
    n_train, n_test = 891, 418
    def make_df(n, with_target=True):
        df = pd.DataFrame({
            'PassengerId': np.arange(1, n+1),
            'Pclass':      np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
            'Name':        [f"Passenger {i}, {np.random.choice(['Mr.','Mrs.','Miss.','Master.','Dr.'], p=[0.57,0.14,0.2,0.05,0.04])} John" for i in range(n)],
            'Sex':         np.random.choice(['male','female'], n, p=[0.65, 0.35]),
            'Age':         np.random.normal(30, 14, n).clip(0.5, 80),
            'SibSp':       np.random.poisson(0.5, n),
            'Parch':       np.random.poisson(0.38, n),
            'Ticket':      [f"T{np.random.randint(1000,9999)}" for _ in range(n)],
            'Fare':        np.random.exponential(32, n).round(2),
            'Cabin':       np.where(np.random.rand(n) < 0.77, np.nan,
                                    [f"{np.random.choice(list('ABCDEF'))}{np.random.randint(1,150)}" for _ in range(n)]),
            'Embarked':    np.random.choice(['S','C','Q'], n, p=[0.72, 0.19, 0.09])
        })
        df.loc[np.random.choice(n, int(n*0.2), replace=False), 'Age'] = np.nan
        if with_target:
            # 타깃 생성 (Sex, Pclass에 강하게 의존)
            logit = (-0.5
                     + 2.5*(df['Sex']=='female').astype(int)
                     - 0.9*(df['Pclass']-1)
                     - 0.02*df['Age'].fillna(30)
                     + 0.1*(df['Fare']/50))
            p = 1/(1+np.exp(-logit))
            df['Survived'] = np.random.binomial(1, p)
        return df

    return make_df(n_train, True), make_df(n_test, False)

train, test = make_titanic()
print("Train:", train.shape, " Test:", test.shape)
train.head()

## 2. 빠른 EDA

In [ ]:
# 결측치
print("[Train 결측]"); print(train.isna().sum()[train.isna().sum() > 0])
print("\n[Test 결측]"); print(test.isna().sum()[test.isna().sum() > 0])

In [ ]:
# 타깃 분포
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
train['Survived'].value_counts().plot(kind='bar', ax=ax[0], color=['salmon','seagreen'])
ax[0].set_title('Survived count'); ax[0].set_xticklabels(['Dead','Alive'], rotation=0)

train.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=ax[1], color='coral')
ax[1].set_title('Survival by Sex')

train.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=ax[2], color='steelblue')
ax[2].set_title('Survival by Pclass')
plt.tight_layout(); plt.show()

## 3. Feature Engineering (핵심)

In [ ]:
def feature_engineering(df):
    df = df.copy()

    # 1. 이름에서 Title 추출 (생존률 강한 상관)
    df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
    title_map = {
        'Mr':'Mr', 'Miss':'Miss', 'Mrs':'Mrs', 'Master':'Master',
        'Dr':'Rare', 'Rev':'Rare', 'Col':'Rare', 'Major':'Rare',
        'Mlle':'Miss', 'Mme':'Mrs', 'Ms':'Miss', 'Lady':'Rare',
        'Sir':'Rare', 'Capt':'Rare', 'Countess':'Rare', 'Don':'Rare',
        'Jonkheer':'Rare'
    }
    df['Title'] = df['Title'].map(title_map).fillna('Rare')

    # 2. 가족 크기
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # 3. 가족 크기 범주
    df['FamilyBin'] = pd.cut(df['FamilySize'], bins=[0,1,4,100], labels=['alone','small','large'])

    # 4. Cabin 유무
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['Deck'] = df['Cabin'].astype(str).str[0]
    df['Deck'] = df['Deck'].replace('n', 'Unknown')

    # 5. Ticket 공유 인원
    ticket_counts = df['Ticket'].value_counts()
    df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)

    # 6. Fare per person
    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize'].clip(lower=1)

    # 7. Age × Pclass (상호작용)
    df['Age*Class'] = df['Age'].fillna(df['Age'].median()) * df['Pclass']

    # 8. Fare 변환
    df['Fare_log'] = np.log1p(df['Fare'])

    # 9. Age 구간
    df['AgeBin'] = pd.cut(df['Age'].fillna(df['Age'].median()),
                          bins=[0, 12, 18, 35, 60, 100],
                          labels=['child','teen','young','adult','senior'])

    # 10. 미취학 아동 플래그
    df['IsChild'] = (df['Age'] < 13).astype(int)

    return df

train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)
print("Shape after FE:", train_fe.shape)
train_fe.head(3)

In [ ]:
# 결측치 처리 - 그룹별 중앙값
for df in [train_fe, test_fe]:
    df['Age']  = df.groupby(['Pclass','Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
    df['Fare'] = df.groupby(['Pclass'])['Fare'].transform(lambda x: x.fillna(x.median()))
    df['Fare_log'] = np.log1p(df['Fare'])
    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize'].clip(lower=1)
    df['Embarked'] = df['Embarked'].fillna('S')

# Fare_log, FarePerPerson 갱신
print("결측치 처리 후:")
print(train_fe.isna().sum()[train_fe.isna().sum()>0])

In [ ]:
# 사용할 피처 선택
drop_cols = ['PassengerId','Name','Ticket','Cabin','Survived']
cat_cols = ['Sex','Embarked','Title','FamilyBin','Deck','AgeBin']

# 범주형을 Label encode
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_fe[col].astype(str), test_fe[col].astype(str)])
    le.fit(combined)
    train_fe[col] = le.transform(train_fe[col].astype(str))
    test_fe[col]  = le.transform(test_fe[col].astype(str))

features = [c for c in train_fe.columns if c not in drop_cols]
print(f"사용할 feature {len(features)}개:")
print(features)

## 4. 모델 학습 (K-Fold)

In [ ]:
X = train_fe[features].values
y = train_fe['Survived'].values
X_test = test_fe[features].values

def train_kfold(model_class, params, X, y, X_test, n_splits=5):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        m = model_class(**params)
        if 'lgb' in model_class.__module__:
            m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        elif 'xgboost' in model_class.__module__:
            m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])], verbose=False)
        else:
            m.fit(X[tr], y[tr])

        oof[val] = m.predict_proba(X[val])[:, 1]
        test_pred += m.predict_proba(X_test)[:, 1] / n_splits

    print(f"{model_class.__name__:25s} OOF AUC: {roc_auc_score(y, oof):.4f}, Acc: {accuracy_score(y, (oof>0.5).astype(int)):.4f}")
    return oof, test_pred

In [ ]:
# LightGBM
lgb_params = dict(n_estimators=1000, learning_rate=0.05, num_leaves=31,
                  max_depth=-1, min_child_samples=20,
                  subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=0.1,
                  random_state=42, verbose=-1, n_jobs=-1)

oof_lgb, test_lgb = train_kfold(lgb.LGBMClassifier, lgb_params, X, y, X_test)

In [ ]:
# XGBoost
xgb_params = dict(n_estimators=1000, learning_rate=0.05, max_depth=5,
                  min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=1.0,
                  early_stopping_rounds=50, eval_metric='auc',
                  random_state=42, n_jobs=-1)

oof_xgb, test_xgb = train_kfold(xgb.XGBClassifier, xgb_params, X, y, X_test)

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_params = dict(n_estimators=500, max_depth=8, min_samples_leaf=2,
                 random_state=42, n_jobs=-1)

oof_rf, test_rf = train_kfold(RandomForestClassifier, rf_params, X, y, X_test)

## 5. 앙상블 (Stacking)

In [ ]:
# Level 0 OOF 예측을 쌓기
oof_stack  = np.column_stack([oof_lgb, oof_xgb, oof_rf])
test_stack = np.column_stack([test_lgb, test_xgb, test_rf])

# Level 1 (Meta): Logistic Regression
meta = LogisticRegression(max_iter=1000, C=1.0)
meta.fit(oof_stack, y)

oof_meta  = meta.predict_proba(oof_stack)[:, 1]
test_meta = meta.predict_proba(test_stack)[:, 1]

print(f"Stacking OOF AUC: {roc_auc_score(y, oof_meta):.4f}")
print(f"Stacking OOF Acc: {accuracy_score(y, (oof_meta>0.5).astype(int)):.4f}")
print(f"Meta coefs: LGB={meta.coef_[0][0]:.3f}, XGB={meta.coef_[0][1]:.3f}, RF={meta.coef_[0][2]:.3f}")

In [ ]:
# Simple Average 비교
test_avg = (test_lgb + test_xgb + test_rf) / 3
oof_avg  = (oof_lgb + oof_xgb + oof_rf) / 3
print(f"Simple Avg OOF Acc: {accuracy_score(y, (oof_avg>0.5).astype(int)):.4f}")

## 6. 제출 파일 생성

In [ ]:
# 최종 예측 (Stacking)
final_pred = (test_meta > 0.5).astype(int)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final_pred
})
submission.to_csv('../data/titanic_submission.csv', index=False)
print(f"제출 파일 생성: {submission.shape}")
submission.head()

### 제출 명령

```bash
kaggle competitions submit -c titanic -f ../data/titanic_submission.csv -m "Stacking: LGBM+XGB+RF with FE"
```

## 7. 개선 아이디어 (더 높은 점수를 위해)

- [ ] Title별 Age 그룹 중앙값으로 결측치 채우기
- [ ] Pseudo-labeling (test 예측을 train에 추가)
- [ ] Target Encoding for Title, Deck, Ticket
- [ ] Optuna로 하이퍼파라미터 튜닝
- [ ] Neural Network 추가
- [ ] CatBoost 추가
- [ ] Seed ensemble (5 seed × 3 model = 15 model 평균)

---

## 📝 Day 22-23 체크리스트
- [ ] EDA → FE → Model → Ensemble → Submit 전체 흐름 체화
- [ ] OOF 방식 K-Fold 구현
- [ ] 3-model stacking 구현
- [ ] submission.csv 생성 및 제출

다음은 **House Prices (회귀)** 대회를 같은 방식으로 풀어봅니다.